In [2]:
import pandas as pd
from scipy import stats

employees = pd.read_csv('employees.csv')
attrition = pd.read_csv('attrition_log.csv')
performance = pd.read_csv('performance.csv')

attrition["exit_date"] = pd.to_datetime(attrition["exit_date"])

In [3]:
employees["is_leaver"] = employees["status"].str.lower() == "departed"

# Dates
employees["hire_date"] = pd.to_datetime(employees["hire_date"])
attrition["exit_date"] = pd.to_datetime(attrition["exit_date"])

# Ordinal performance scale
RATING_ORDER = {
    "Unsatisfactory": 1,
    "Below Expectations": 2,
    "Meets Expectations": 3,
    "High Performer": 4,
    "Outstanding": 5,
}
performance["rating_score"] = performance["performance_rating"].map(RATING_ORDER)
assert performance["rating_score"].isna().sum() == 0, "Unmapped rating label found"

print("is_leaver value counts:")
print(employees["is_leaver"].value_counts())

is_leaver value counts:
is_leaver
False    12003
True      1400
Name: count, dtype: int64


In [4]:
print(employees["department"].unique())

['Wealth Management' 'Insurance' 'Corporate Operations' 'Technology'
 'Retail Banking' 'Risk & Compliance' 'Executive Leadership']


In [5]:
DEPT_NAME = "Risk & Compliance"
rc_employees = employees[employees["department"] == DEPT_NAME].copy()
print(f"{DEPT_NAME} headcount (all-time, incl. leavers): {len(rc_employees)}")
print(rc_employees["is_leaver"].value_counts())

Risk & Compliance headcount (all-time, incl. leavers): 2022
is_leaver
False    1783
True      239
Name: count, dtype: int64


In [6]:
print(attrition.columns.tolist())

print(attrition.index)
print(attrition.columns.tolist())

['employee_id', 'exit_date', 'exit_type', 'stated_exit_reason', 'notice_period_served', 'regrettable_flag', 'performance_band_at_exit', 'salary_at_exit', 'manager_id_at_exit', 'pathway']
RangeIndex(start=0, stop=1400, step=1)
['employee_id', 'exit_date', 'exit_type', 'stated_exit_reason', 'notice_period_served', 'regrettable_flag', 'performance_band_at_exit', 'salary_at_exit', 'manager_id_at_exit', 'pathway']


In [7]:
rc_leavers = rc_employees[rc_employees["is_leaver"]].merge(
    attrition[["employee_id", "exit_date", "exit_type", "regrettable_flag", "stated_exit_reason", "pathway"]],
    on="employee_id", how="left"
)

# print("exit_date" in employees.columns)
# print(rc_leavers.columns.tolist())

# compare = rc_leavers[["employee_id", "exit_date_x", "exit_date_y"]]
# print(compare.head(10))
# print("\nRows where they differ:", (compare["exit_date_x"] != compare["exit_date_y"]).sum())

# compare = rc_leavers[["employee_id", "exit_date_x", "exit_date_y"]]
# print(compare.head(10))
# print("\nRows where they differ:", (compare["exit_date_x"] != compare["exit_date_y"]).sum())

In [8]:
rc_leavers = rc_employees[rc_employees["is_leaver"]].drop(columns=["exit_date"], errors="ignore").merge(
    attrition[["employee_id", "exit_date", "exit_type", "regrettable_flag", "stated_exit_reason", "pathway"]],
    on="employee_id", how="left"
)

rc_leavers["exit_date"] = pd.to_datetime(rc_leavers["exit_date"])
rc_leavers["hire_date"] = pd.to_datetime(rc_leavers["hire_date"])

rc_leavers["working_days"] = (rc_leavers["exit_date"] - rc_leavers["hire_date"]).dt.days

print(f"R&C leavers: {len(rc_leavers)}")
print("\nWorking days summary:")
print(rc_leavers["working_days"].describe())

missing_exit = rc_leavers["exit_date"].isna().sum()
if missing_exit:
    print(f"\nWARNING: {missing_exit} R&C leaver(s) have no matching attrition record — working_days is NaN for these.")

R&C leavers: 239

Working days summary:
count      239.000000
mean      4185.640167
std       4457.856854
min         24.000000
25%        254.500000
50%       2526.000000
75%       7912.500000
max      13594.000000
Name: working_days, dtype: float64


In [9]:
print("Stated exit reasons (count):")
print(rc_leavers["stated_exit_reason"].value_counts())

print("\nStated exit reasons (%):")
print(rc_leavers["stated_exit_reason"].value_counts(normalize=True).mul(100).round(1))

print("\nExit type (voluntary/involuntary):")
print(rc_leavers["exit_type"].value_counts(normalize=True).mul(100).round(1))

print("\nRegrettable vs non-regrettable:")
print(rc_leavers["regrettable_flag"].value_counts(normalize=True).mul(100).round(1))

Stated exit reasons (count):
stated_exit_reason
Career advancement                   113
Better opportunity                    51
Involuntary - performance             33
Work-life balance                      8
Personal reasons                       7
Involuntary - conduct                  6
Involuntary - restructure              5
Study/career change                    5
Compensation                           4
Role uncertainty / unclear future      4
Relocation                             3
Name: count, dtype: int64

Stated exit reasons (%):
stated_exit_reason
Career advancement                   47.3
Better opportunity                   21.3
Involuntary - performance            13.8
Work-life balance                     3.3
Personal reasons                      2.9
Involuntary - conduct                 2.5
Involuntary - restructure             2.1
Study/career change                   2.1
Compensation                          1.7
Role uncertainty / unclear future     1.7
Relocation

In [10]:
performance["review_date"] = pd.to_datetime(performance["review_date"])

last_review = (
    performance.sort_values("review_date")
    .groupby("employee_id")
    .tail(1)[["employee_id", "performance_rating", "rating_score"]]
)

rc_leavers_perf = rc_leavers.merge(last_review, on="employee_id", how="left")

missing_perf = rc_leavers_perf["rating_score"].isna().sum()
if missing_perf:
    print(f"NOTE: {missing_perf} R&C leaver(s) have no performance review on record.")

print("\nLast rating distribution among R&C leavers (%):")
rating_order_cols = ["Unsatisfactory", "Below Expectations", "Meets Expectations", "High Performer", "Outstanding"]
dist = rc_leavers_perf["performance_rating"].value_counts(normalize=True).mul(100).round(1)
print(dist.reindex(rating_order_cols))


Last rating distribution among R&C leavers (%):
performance_rating
Unsatisfactory         3.8
Below Expectations     9.6
Meets Expectations    42.7
High Performer        29.7
Outstanding           14.2
Name: proportion, dtype: float64


In [11]:
# High performer or above = rating_score >= 4 ("High Performer" or "Outstanding")
HIGH_PERFORMER_THRESHOLD = 4

n_high_leavers = (rc_leavers_perf["rating_score"] >= HIGH_PERFORMER_THRESHOLD).sum()
n_total_leavers_rated = rc_leavers_perf["rating_score"].notna().sum()

print(f"R&C leavers who are High Performer or above: {n_high_leavers} of {n_total_leavers_rated} rated "
      f"({n_high_leavers / n_total_leavers_rated * 100:.1f}%)")

R&C leavers who are High Performer or above: 105 of 239 rated (43.9%)


In [12]:
rc_all_perf = rc_employees.merge(last_review, on="employee_id", how="left")
rc_all_perf["is_high_performer"] = rc_all_perf["rating_score"] >= HIGH_PERFORMER_THRESHOLD

print("High-performer rate, R&C leavers vs stayers (%):")
print(rc_all_perf.groupby("is_leaver")["is_high_performer"].mean().mul(100).round(1))

ct_high = pd.crosstab(rc_all_perf["is_leaver"], rc_all_perf["is_high_performer"])
print("\nCounts:")
print(ct_high)

if ct_high.shape == (2, 2):
    chi2_h, p_high, dof_h, exp_h = stats.chi2_contingency(ct_high)
    print(f"\nChi-square (is_leaver x is_high_performer): chi2={chi2_h:.2f}, p={p_high:.4f}")
    if p_high < 0.05:
        print("  -> Statistically significant: high-performer rate differs between R&C leavers and stayers.")
    else:
        print("  -> Not statistically significant at alpha=0.05.")
else:
    print("\nNot enough variation in one group to run chi-square (check cell counts above).")

High-performer rate, R&C leavers vs stayers (%):
is_leaver
False    38.6
True     43.9
Name: is_high_performer, dtype: float64

Counts:
is_high_performer  False  True 
is_leaver                      
False               1095    688
True                 134    105

Chi-square (is_leaver x is_high_performer): chi2=2.31, p=0.1287
  -> Not statistically significant at alpha=0.05.


In [13]:
LOW_PERFORMER_THRESHOLD = 2

rc_all_perf["is_low_performer"] = rc_all_perf["rating_score"] <= LOW_PERFORMER_THRESHOLD

print("Low-performer rate, R&C leavers vs stayers (%):")
print(rc_all_perf.groupby("is_leaver")["is_low_performer"].mean().mul(100).round(1))

ct_low = pd.crosstab(rc_all_perf["is_leaver"], rc_all_perf["is_low_performer"])
print("\nCounts:")
print(ct_low)

if ct_low.shape == (2, 2):
    chi2_l, p_low, dof_l, exp_l = stats.chi2_contingency(ct_low)
    print(f"\nChi-square (is_leaver x is_low_performer): chi2={chi2_l:.2f}, p={p_low:.4f}")
    if p_low < 0.05:
        print("  -> Statistically significant: low-performer rate differs between R&C leavers and stayers.")
    else:
        print("  -> Not statistically significant at alpha=0.05.")
else:
    print("\nNot enough variation in one group to run chi-square (check cell counts above).")

Low-performer rate, R&C leavers vs stayers (%):
is_leaver
False    15.4
True     13.4
Name: is_low_performer, dtype: float64

Counts:
is_low_performer  False  True 
is_leaver                     
False              1508    275
True                207     32

Chi-square (is_leaver x is_low_performer): chi2=0.53, p=0.4672
  -> Not statistically significant at alpha=0.05.


In [ ]:
summary = pd.DataFrame([
    {"metric": "R&C leavers - median working days", "value": rc_leavers["working_days"].median()},
    {"metric": "R&C leavers - % High Performer or above", "value": round(n_high_leavers / n_total_leavers_rated * 100, 1)},
    {"metric": "R&C - high-performer rate (stayers, %)", "value": round(rc_all_perf.loc[~rc_all_perf['is_leaver'], 'is_high_performer'].mean() * 100, 1)},
    {"metric": "R&C - high-performer rate (leavers, %)", "value": round(rc_all_perf.loc[rc_all_perf['is_leaver'], 'is_high_performer'].mean() * 100, 1)},
    {"metric": "R&C - low-performer rate (stayers, %)", "value": round(rc_all_perf.loc[~rc_all_perf['is_leaver'], 'is_low_performer'].mean() * 100, 1)},
    {"metric": "R&C - low-performer rate (leavers, %)", "value": round(rc_all_perf.loc[rc_all_perf['is_leaver'], 'is_low_performer'].mean() * 100, 1)},
])
print(summary.to_string(index=False))

                                 metric  value
      R&C leavers - median working days 2526.0
R&C leavers - % High Performer or above   43.9
 R&C - high-performer rate (stayers, %)   38.6
 R&C - high-performer rate (leavers, %)   43.9
  R&C - low-performer rate (stayers, %)   15.4
  R&C - low-performer rate (leavers, %)   13.4
